# AX 1주차 쇼핑몰 데이터 분석 프로젝트 - 데이터 로더(Role A) 회고

1. 내가 이 프로젝트에서 기여한 점

•	데이터 로딩 및 전처리 모듈화 (src/data_loader.py)

•	원본 CSV 4종(customers, products, orders, order_details)의 연관 관계(PK/FK)를 파악하고, 팀원이 재사용할 수 있도록 함수화했습니다.

•	문자열(object) 타입의 날짜 컬럼을 시계열 연산이 가능한 datetime64 구조로 변환했습니다.

In [ ]:
# src/data_loader.py
import pandas as pd

def load_and_merge_data(data_dir: str = "../data/raw") -> pd.DataFrame:
    """4개 CSV 데이터를 로드하고 날짜 표준화 후 병합하는 함수"""
    customers = pd.read_csv(f"{data_dir}/customers.csv")
    orders = pd.read_csv(f"{data_dir}/orders.csv")

    # 날짜 데이터 표준화
    orders['order_date'] = pd.to_datetime(orders['order_date'])
    customers['signup_date'] = pd.to_datetime(customers['signup_date'])

    # 데이터 병합 (customer_id 기준)
    merged_df = pd.merge(orders, customers, on='customer_id', how='left')
    return merged_df

•	데이터 품질 전수 검사 (notebooks/01_data_check.ipynb)

•	결측치 및 중복 데이터를 무조건 삭제하지 않고, 비즈니스 영향도를 검토하여 보존 여부를 판단했습니다.

In [ ]:
# 데이터 품질 검증 4단계 스크립트
print("1. 데이터 크기(행, 열):", df.shape)
print("\n2. 컬럼 정보 및 타입:")
df.info()
print("\n3. 결측치 수량:\n", df.isna().sum())
print("\n4. 중복 데이터 수:", df.duplicated().sum())

•	AI 코드 검증 절차 실행
•	AI가 제안한 코드를 5줄 이하 단위로 분할 실행하며 예상 결과와 실제 출력 결과를 3회 이상 비교 검증했습니다.

2. 프로젝트 진행 중 발생한 문제 (Challenges)
	
    1.	FileNotFoundError: VS Code 작업 루트 디렉토리와 주피터 노트북 파일 위치(notebooks/) 간의 실행 기준점이 달라 상대 경로 참조 실패.
	
    2.	NameError: 셀 실행 순서가 뒤얽히거나 커널 재시작 시 메모리 내 변수 미선언 오류 발생.

3. 문제 해결 지점 (Troubleshooting)

•	경로 인식 오류 해결 (os.getcwd() 체킹)

•	현재 파이썬 프로세스가 바라보는 위치를 파악하고, 상위 디렉토리 이동 기호(../)를 적용하여 경로 추적.

In [ ]:
import os

# 1. 현재 실행 디렉토리 위치 확인
print("현재 실행 위치:", os.getcwd())

# 2. 상대 경로 명시 및 존재 여부 검증
data_path = "../data/raw/customers.csv"
if os.path.exists(data_path):
    customers = pd.read_csv(data_path)
    print("데이터 로드 성공")
else:
    print("경로가 올바르지 않습니다.")

•	순차 실행 구조 재설계

•	데이터 로드와 검증 코드를 단일 파이프라인으로 구성하여 커널 상태 변동에 따른 에러 방지.

In [ ]:
# 독립 실행 가능한 로드 및 연산 블록
try:
    orders = pd.read_csv("../data/raw/orders.csv")
    orders['order_date'] = pd.to_datetime(orders['order_date'])
    print(f"주문 데이터 로드 완료: 총 {len(orders)}건")
except NameError as e:
    print("선행 실행 셀이 누락되었습니다:", e)

4. 어려웠던 점 & 향후 학습 계획 (Keep & Try)

•	OS 독립적 경로 관리 (pathlib 활용)

•	운영체제별 경로 구분자(\ vs /) 문제 방지를 위해 파이썬 표준 라이브러리 pathlib 적용 학습 필요.

•	Git/GitHub 충돌 해결 절차 체화

•	브랜치 생성 및 PR을 넘어, 코드 충돌(Conflict) 발생 시 안전하게 병합하는 커맨드 학습 예정.

•	Pandas 데이터 가공 고도화

•	단순 조회 수준을 넘어 loc, iloc, apply를 활용한 조건부 변환 연산 숙달 예정.